# C9-dimensionality-reduction — Review

Work through this notebook *after* the three lesson sessions and
(ideally) the practice sets.
It is a consolidation tool: a summary table, the idiom sheet, a
self-quiz, and pointers to what to redo.
Quiz answers are collapsed at the very end — commit to your answers
before looking.

In [ ]:
import numpy as np

## Concept summary

| Concept | One-line summary | Key fact to retain |
|---|---|---|
| PCA | Center, then SVD: $X_c = U\Sigma V^{\mathsf T}$; directions = rows of $V^{\mathsf T}$, component variances $= \sigma_i^2/(n-1)$, scores $T = X_c V = U\Sigma$ | the pinned route has no covariance detour; always assert $X_c$'s column means $\approx 0$; `np.var` needs `ddof=1`; evr $= \sigma_i^2/\sum\sigma_j^2$ sums to exactly 1 |
| Sign ambiguity | $v$ and $-v$ are the same axis — SVD/eigh may return either | compare individual directions via $\lvert v\cdot w\rvert$ or sign-fix (largest-$\lvert\cdot\rvert$ entry made positive, F6-02's pin); variances and distances never feel the flip |
| Truncated SVD (practice) | $W_r = U_{:,:r}\Sigma_r V^{\mathsf T}_{:r}$: best rank-$r$ fit; error from the spectrum | $\lVert W-W_r\rVert_F^2 = \sum_{i>r}\sigma_i^2$ — cheap error curves via `cumsum`; unit-row stacks have $\sum\sigma_i^2 = N$; budget choice $r^\*$ needs the two-sided certificate; $S_r$'s diagonal sags below 1 |
| umap-concept | Neighbor-graph layout: $k$-NN graph in the original space, arranged so graph neighbors stay close | local neighborhoods preserved, global distances/densities/sizes distorted; axes carry no units or feature meaning; settings and randomness change layouts; concept only — no library in this unit |
| Local vs global structure | Different views answer different questions; both sides are measurable | $k$-NN preservation (local), worst-case stretch (global); a projection never exaggerates distances (far-in-view ⇒ far-in-reality) but manufactures false neighbors on curved data |

## Formula and idiom sheet

**By hand (the forms the exam's registers expect):**

- Mean vector $\mu = \frac1n\sum_i X_i$; centered $X_c = X - \mu$;
  every column of $X_c$ has mean exactly $0$.
- Directional variance (unit $u$):
  $\lVert X_c u\rVert^2/(n-1)$; maximized by $v_1$.
- Component variances $\sigma_i^2/(n-1)$; total variance
  $= \lVert X_c\rVert_F^2/(n-1) = \sum_i \sigma_i^2/(n-1)$
  (two routes, one number).
- $\mathrm{evr}_i = \sigma_i^2/\sum_j \sigma_j^2$ (the $n-1$ cancels);
  $\sum_i \mathrm{evr}_i = 1$.
- Truncation error: $\lVert W - W_r\rVert_F^2 = \sum_{i>r}\sigma_i^2$;
  on the similarity matrix:
  $\lVert S - S_r\rVert_F^2 = \sum_{i>r}\sigma_i^4$ (p12), with
  $\lVert S\rVert_F^2 = \sum_i \sigma_i^4$.
- Hand-SVD of small matrices: eigenvalues of the Gram matrix are the
  $\sigma_i^2$ (F6's route; the $2\times2$ eigen condition).

**NumPy idioms (this unit's register):**

```python
mu = X.mean(axis=0); Xc = X - mu                     # center, then assert:
assert np.max(np.abs(Xc.mean(axis=0))) < 1e-10
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)    # s already descending
comp_var = s**2 / (n - 1)                            # component variances
evr = s**2 / (s**2).sum()                            # explained ratios
T2 = Xc @ Vt[:2].T                                   # 2-D scores/view
v_fixed = v * np.sign(v[np.argmax(np.abs(v))])       # the sign pin
Wr = U[:, :r] @ np.diag(s[:r]) @ Vt[:r]              # rank-r truncation
rel_err2 = (fro2 - np.cumsum(s**2)) / fro2           # the error curve
r_star = int(np.argmax(rel_err2 <= B) + 1)           # budget choice, plus:
assert rel_err2[r_star - 1] <= B and (r_star == 1 or rel_err2[r_star - 2] > B)
```

Standing habits: assert near-zero column means before every SVD-of-$X_c$;
verify every truncation against the $\sigma$-tail; `ddof=1` for sample
variances; `np.isclose(..., atol=<stated>, rtol=0)` for anchors; never
compare raw signed eigenvectors across routes.

## Self-quiz

Twelve items, all four concepts covered.
Work by hand (calculator-free), then check against the collapsed
answers at the very end.

1. Recite the pinned PCA route in four steps, and name the two numbers
   you read off the SVD (per component).
2. Raw points $(3, 1), (1, 3), (5, 3), (3, 5)$: compute $\mu$ and the
   centered rows.
3. The centered matrix of a $4$-point dataset has singular values
   $\sigma = (4, 2)$. Give the component variances, the evr pair, and
   PC1's evr in normal form $p/q$ with $p + q$.
4. Your PC1 is $(0.6, -0.8)$; a teammate's is $(-0.6, 0.8)$.
   Are you in disagreement? Which two comparison idioms settle it?
5. A unit-row stack has $N = 32$ rows. What is $\sum_i \sigma_i^2$,
   and why?
6. Write the one-line error-curve idiom, and state which index of
   `rel_err2` holds rank $r$'s error.
7. A claimed budget answer says $r^\* = 12$ for $B = 0.10$.
   What *two* asserts certify it?
8. Why does $S_r = W_r W_r^{\mathsf T}$ have diagonal entries $< 1$,
   and what false invariant would `assert np.allclose(np.diag(Sr), 1.0)`
   import?
9. State umap-concept's four stated facts in one clause each.
10. A UMAP-style map shows cluster A $2$ cm from B and $6$ cm from C.
    What may you conclude about the data, and what not?
11. Define $k$-NN preservation, give chance level for $n = 600$,
    $k = 10$, and recall the ribbon's three scores (unrolled, PCA-2,
    PCA-1).
12. State the projection guarantee and its one-sided inference rule —
    which direction of conclusion is licensed, which is the trap?

## What to redo, per weak spot

| If you struggled with… | Redo (practice) | Reread (lesson) |
|---|---|---|
| items 1–3 (route, centering, evr arithmetic) | p01, p04, p05, p11 | Session 1 §§1–3, §5, §7 |
| item 4 (sign discipline) | p13, p01 | Session 1 §4, §8 |
| items 5–7 (spectra, error curves, budgets) | p06, p07, p09, p14 | Session 2 §§2–5, §7 |
| item 8 (what compression breaks) | p12, p14 | Session 2 §6, §8 |
| items 9–10 (umap-concept, reading maps) | p02, p15, p16, p17 | Session 3 §4, §7, §8 |
| items 11–12 (metrics, the guarantee) | p03, p08, p10, p18 | Session 3 §§3, 5–6 |
| integration under pressure | p13, p14, p17, p18 | — |

---

Scored yourself below ~9/12? Use the redo table above, then retake the
quiz.
At 10+ you are ready for **`C10-competition-craft`** — the course's
final unit, where the entire C1–C9 toolkit runs under competition
conditions: the graded-notebook contract, hidden-test discipline, and
metric-driven iteration.

## Self-quiz answers

<details><summary><b>Click to reveal (commit to your answers first)</b></summary>

1. Center ($\mu$, $X_c = X - \mu$) → `np.linalg.svd(Xc,
   full_matrices=False)` → directions = rows of $V^{\mathsf T}$ →
   per component: the variance $\sigma_i^2/(n-1)$ and the share
   $\mathrm{evr}_i = \sigma_i^2/\sum\sigma_j^2$.
2. $\mu = (3, 3)$; centered rows
   $(0, -2), (-2, 0), (2, 0), (0, 2)$.
3. Variances $16/3$ and $4/3$; evr $= (16/20, 4/20) = (4/5, 1/5)$;
   normal form $4/5$, $p + q = 9$.
4. No disagreement: the vectors differ by a global sign — the same
   axis. $\lvert v \cdot w\rvert \approx 1$, or sign-fix both (the
   largest-magnitude entry is the $0.8$ slot, so both become
   $(-0.6, 0.8)$) and compare with `np.allclose` at a stated
   `atol`, `rtol=0`.
5. $32$: each unit row contributes $1$ to
   $\lVert W\rVert_F^2 = \sum_i \sigma_i^2$.
6. `rel_err2 = (fro2 - np.cumsum(s**2)) / fro2`; rank $r$ lives at
   `rel_err2[r - 1]`.
7. `assert rel_err2[11] <= 0.10` (meets the budget) and
   `assert rel_err2[10] > 0.10` (minimally — $r^\* - 1$ fails).
8. $(S_r)_{ii} = \lVert (W_r)_i\rVert^2$, and truncation only shortens
   rows (dropped orthonormal coordinates). The assert would import
   the *uncompressed* world's unit-diagonal invariant, which is false
   after truncation.
9. (1) local neighborhoods first; (2) global distances, densities,
   and sizes carry no quantitative meaning; (3) axes have no units
   and no feature recipes; (4) settings and randomness change the
   layout.
10. May conclude: that tight co-location *within* a clump suggests
    genuine neighbors (stated fact 1's supported reading). May not
    conclude: that A is "three times more similar" to B than to C,
    or any quantitative distance/size claim — stated fact 2.
11. For each point, the fraction of its $k$ original-space nearest
    neighbors that remain among its $k$ view-space nearest neighbors,
    averaged. Chance $\approx k/(n-1) = 10/599 \approx 0.017$.
    Ribbon: unrolled $0.9322$, PCA top-2 $0.5877$, PCA top-1
    $0.1303$.
12. A projection onto orthonormal axes never increases any pairwise
    distance. Licensed: far-in-view ⇒ far-in-reality. The trap:
    close-in-view ⇒ close-in-reality (false neighbors — the ribbon's
    $0.0231$-apart pair was $1.9532$ apart).

</details>